# Conversational App for Itinerary Planning

In [1]:
import os
from dotenv import load_dotenv
#from langchain import HuggingFaceHub

from langchain import PromptTemplate, LLMChain

load_dotenv()

#os.environ["HUGGINGFACEHUB_API_TOKEN"]
#retrieving api key
key=os.environ['OPENAI_API_KEY']

In [2]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage
)
from langchain.chains import LLMChain, ConversationChain
from langchain_openai import ChatOpenAI

chat = ChatOpenAI()

## Sample bot with no memory

In [3]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage
)
from langchain.chains import LLMChain, ConversationChain
from langchain_openai import ChatOpenAI

chat = ChatOpenAI()
messages = [
    SystemMessage(content="You are a helpful assistant that help the user to plan an optimized itinerary."),
    HumanMessage(content="I'm going to Rome for 2 days, what can I visit?")
]

# Use invoke instead of the deprecated __call__ method
output = chat.invoke(messages)
print(output.content)

In Rome, there are many amazing attractions to visit. Here is an optimized itinerary for your 2-day trip to Rome:

Day 1:
1. Start your day early by visiting the iconic Colosseum, the largest amphitheater ever built in the Roman Empire.
2. Explore the Roman Forum, located near the Colosseum, where you can see ancient ruins and learn about ancient Roman life.
3. Visit the Pantheon, a well-preserved ancient Roman temple dedicated to the gods.
4. Enjoy a relaxing walk through Piazza Navona, a charming square with beautiful fountains and street performers.
5. End your day by tossing a coin into the Trevi Fountain for good luck and take some time to admire this stunning Baroque masterpiece.

Day 2:
1. Visit Vatican City and explore St. Peter's Basilica, one of the largest churches in the world with impressive architecture and art.
2. Enter the Vatican Museums to see the famous Sistine Chapel, decorated by Michelangelo, and other art collections.
3. Walk through the picturesque neighborhood 

## Adding Memory

In [4]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from typing import List

# Initialize the chat model
chat = ChatOpenAI()

# Create a message history
message_history = ChatMessageHistory()

# Add system message
system_message = SystemMessage(content="You are a helpful assistant.")
message_history.add_message(system_message)

# Create a messages list
messages = message_history.messages + [
    HumanMessage(content="Hi there!")
]

# Run the conversation
response = chat.invoke(messages)

# Add the response to history
message_history.add_message(response)

In [5]:
# Add a new human message for the follow-up question
new_question = HumanMessage(content="What is the most iconic place in Rome?")
message_history.add_message(new_question)

# Get all messages from the history
messages = message_history.messages

# Invoke the chat model with all messages
response = chat.invoke(messages)

# Add the response to history
message_history.add_message(response)

# Print the response
print(response.content)

One of the most iconic places in Rome is the Colosseum. It is a large ancient amphitheater that was used for gladiatorial contests and other public spectacles. The Colosseum is a symbol of ancient Rome and is one of the most visited tourist attractions in the city.


In [6]:
# Add a new human message for the follow-up question
new_question = HumanMessage(content="What kind of other events?")
message_history.add_message(new_question)

# Get all messages from the history
messages = message_history.messages

# Invoke the chat model with all messages
response = chat.invoke(messages)

# Add the response to history
message_history.add_message(response)

# Print the response
print(response.content)

In addition to gladiatorial contests, the Colosseum was also used for other public spectacles and events such as animal hunts, mock sea battles, executions, and theatrical performances. It was a center of entertainment and social gathering for the people of ancient Rome.


In [7]:
# Get all messages from the message history
messages = message_history.messages

In [8]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_community.chat_message_histories import ChatMessageHistory  # Try community package
from langchain_core.runnables import RunnableSequence
from langchain_openai import ChatOpenAI

# Initialize the chat model if not already done
chat = ChatOpenAI()

# Keep your prompt setup the same
prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "You are a helpful assistant that help the user to plan an optimized itinerary."
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    HumanMessagePromptTemplate.from_template("{question}")
])

# Set up message history instead of memory
chat_history = ChatMessageHistory()

# Create the chain using the pipe syntax
conversation = prompt | chat

# To use it with chat history, create an invoke function
async def invoke_conversation(question: str):
    # Get messages from chat history
    messages = chat_history.messages
    
    # Invoke conversation
    response = await conversation.ainvoke({
        "chat_history": messages,
        "question": question
    })
    
    # Add the new messages to history
    chat_history.add_user_message(question)
    chat_history.add_ai_message(response.content)
    
    return response

# Non-async version if needed
def invoke_conversation_sync(question: str):
    # Get messages from chat history
    messages = chat_history.messages
    
    # Invoke conversation
    response = conversation.invoke({
        "chat_history": messages,
        "question": question
    })
    
    # Add the new messages to history
    chat_history.add_user_message(question)
    chat_history.add_ai_message(response.content)
    
    return response

In [9]:
while True:
    query = input('you: ')
    if query == 'q':
        break
    
    # Load memory contents
    memory_contents = memory.load_memory_variables({})
    
    # Use invoke() method instead of calling directly
    output = conversation.invoke({
        "chat_history": memory_contents["chat_history"],
        "question": query
    })
    
    # The output structure is different in the new format
    print('User: ', query)
    print('AI system: ', output.content)  # Use .content instead of ['text']
    
    # Save the conversation to memory
    memory.save_context({"question": query}, {"output": output.content})

you:  q


## Adding non parametric knowledge

In [10]:
from langchain.llms import OpenAI
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.document_loaders import PyPDFLoader

import os

from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"]

text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1500,
            chunk_overlap=200
        )

raw_documents = PyPDFLoader('italy_travel.pdf').load()
documents = text_splitter.split_documents(raw_documents)
db = FAISS.from_documents(documents, OpenAIEmbeddings())

In [25]:
# Updated imports - try langchain_community.retrievers.conversational_retrieval_chain
from langchain_community.retrievers.conversational_retrieval_chain import ConversationalRetrievalChain
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_openai import ChatOpenAI

# Create a message history to store the conversation
message_history = ChatMessageHistory()

# Initialize the LLM
llm = ChatOpenAI()

# Create the chain with the message history
qa_chain = ConversationalRetrievalChain.from_llm(
    llm, 
    retriever=db.as_retriever(), 
    memory_key='chat_history',
    return_source_documents=True,
    get_chat_history=lambda h: message_history.messages
)

# Use invoke instead of run
response = qa_chain.invoke({"question": "Give me some review about the Pantheon"})

# Add the question and answer to the message history
message_history.add_user_message("Give me some review about the Pantheon")
message_history.add_ai_message(response["answer"])

# Print the answer
print(response["answer"])

ModuleNotFoundError: No module named 'langchain_community.retrievers.conversational_retrieval_chain'

In [12]:
from langchain.prompts.prompt import PromptTemplate

custom_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question. 
If you cannot find the answer in the document provided, ignore the document and answer anyway.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

CUSTOM_QUESTION_PROMPT = PromptTemplate.from_template(custom_template)

memory = ConversationBufferMemory(
            memory_key='chat_history',
            return_messages=True
        )


qa_chain = ConversationalRetrievalChain.from_llm(llm, retriever=db.as_retriever(), condense_question_prompt=CUSTOM_QUESTION_PROMPT, memory=memory, verbose=True)
qa_chain.run({'question':'What can I visit in India?'})



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
What to see
 in Italy
Page 6
Canals of Venice 
la peñita:
 To visit Venice in January is to discover another city different from that found on other occasions in July or August.
Tourists (from myself included) inadvertently "spoil" the best views of the sites. If in summer the city is abuzz with activity, in winter
it has a romantic touch that makes me like it even more, if possible. Getting lost in the alleys away from the sea of humanity,
finding a niche with an carved image, a passageway under a building, or a hidden courtyard awaken in my mind an adventurous
sensations which makes me like Venice most each time I visit. Also, you can get off the beaten track and mix it up with the
inhabitants of t

"I don't know."

In [13]:
from langchain.agents.agent_toolkits import create_retriever_tool

tool = create_retriever_tool(
    db.as_retriever(), 
    "italy_travel",
    "Searches and returns documents regarding Italy."
)
tools = [tool]

memory = ConversationBufferMemory(
            memory_key='chat_history',
            return_messages=True
        )

from langchain.agents.agent_toolkits import create_conversational_retrieval_agent

from langchain.chat_models import ChatOpenAI
llm = ChatOpenAI(temperature = 0)

agent_executor = create_conversational_retrieval_agent(llm, tools, memory_key='chat_history', verbose=True)

/var/folders/5s/f9f7n_tj38dfzkc2lkzw67q80000gn/T/ipykernel_60990/437833383.py:18: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(temperature = 0)


In [14]:
agent_executor.invoke({"input": "hi, i'm Vale"})



> Entering new AgentExecutor chain...
Hello Vale! How can I assist you today?

> Finished chain.


{'input': "hi, i'm Vale",
 'chat_history': [HumanMessage(content="hi, i'm Vale", additional_kwargs={}, response_metadata={}),
  AIMessage(content='Hello Vale! How can I assist you today?', additional_kwargs={}, response_metadata={})],
 'output': 'Hello Vale! How can I assist you today?',
 'intermediate_steps': []}

In [15]:
agent_executor.invoke({"input": "Tell me something about Pantheon"})



> Entering new AgentExecutor chain...

Invoking: `italy_travel` with `{'query': 'Pantheon'}`


cafes in the square. The most famous are the Quadri and
Florian. 
Piazza San Marco, 
Venice
4
Historical Monuments
Pantheon
 
Miskita:
 
"Angelic and non-human design," was how
Michelangelo described the Pantheon 14 centuries after its
construction. The highlights are the gigantic dome, the upper
eye, the sheer size of the place, and the harmony of the
whole building. We visited with a Roman guide which is
exactly how you should (or shouldn't) visit the city, especially
since they talk too much and have lots and lots of history of
each building. 
And so we learned things like how the interior
dome was filled with sand during construction. Or not,
because it really is impossible to be sure of anything when
one doesn't know Italian and the guide has a very thick
Roman accent. But we loved the place, especially the
entrance portico with its stout columns. 
 
Almudena:
 
The Pantheon is one of 

{'input': 'Tell me something about Pantheon',
 'chat_history': [HumanMessage(content="hi, i'm Vale", additional_kwargs={}, response_metadata={}),
  AIMessage(content='Hello Vale! How can I assist you today?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Tell me something about Pantheon', additional_kwargs={}, response_metadata={}),
  AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"query":"Pantheon"}', 'name': 'italy_travel'}}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 110, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'function_call', 'logprobs': None}, id='run-410daf18-fd6b-4868-9e11-448395133418-0'),
  FunctionMessage(content='cafes in the square.

In [16]:
output = agent_executor({"input": "what can I visit in India in 3 days?"})
output['output']



> Entering new AgentExecutor chain...


/var/folders/5s/f9f7n_tj38dfzkc2lkzw67q80000gn/T/ipykernel_60990/180987740.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  output = agent_executor({"input": "what can I visit in India in 3 days?"})


In India, with only 3 days to explore, you can consider visiting some of the following popular destinations:

1. Delhi: Explore the historical landmarks such as Red Fort, India Gate, and Qutub Minar. Visit the bustling markets like Chandni Chowk and Connaught Place. Don't miss out on experiencing the vibrant culture and delicious street food of Delhi.

2. Agra: Visit the iconic Taj Mahal, one of the Seven Wonders of the World. Explore the Agra Fort and enjoy the Mughal architecture in the city. A day trip to Agra from Delhi is feasible and allows you to witness the beauty of the Taj Mahal.

3. Jaipur: Known as the Pink City, Jaipur is famous for its majestic forts and palaces. Visit the Amber Fort, City Palace, Hawa Mahal, and Jantar Mantar. Explore the local markets for traditional handicrafts and textiles.

These destinations offer a glimpse into India's rich history, culture, and architecture within a short timeframe. Remember to plan your itinerary efficiently to make the most of y

"In India, with only 3 days to explore, you can consider visiting some of the following popular destinations:\n\n1. Delhi: Explore the historical landmarks such as Red Fort, India Gate, and Qutub Minar. Visit the bustling markets like Chandni Chowk and Connaught Place. Don't miss out on experiencing the vibrant culture and delicious street food of Delhi.\n\n2. Agra: Visit the iconic Taj Mahal, one of the Seven Wonders of the World. Explore the Agra Fort and enjoy the Mughal architecture in the city. A day trip to Agra from Delhi is feasible and allows you to witness the beauty of the Taj Mahal.\n\n3. Jaipur: Known as the Pink City, Jaipur is famous for its majestic forts and palaces. Visit the Amber Fort, City Palace, Hawa Mahal, and Jantar Mantar. Explore the local markets for traditional handicrafts and textiles.\n\nThese destinations offer a glimpse into India's rich history, culture, and architecture within a short timeframe. Remember to plan your itinerary efficiently to make the 

## Adding external tools

In [17]:
from langchain import SerpAPIWrapper
from langchain.agents import AgentType, initialize_agent
from langchain.llms import OpenAI
from langchain.tools import BaseTool, StructuredTool, Tool, tool

import os
from dotenv import load_dotenv

load_dotenv()

key = os.environ["SERPAPI_API_KEY"]

search = SerpAPIWrapper()

In [18]:
tools = [
    Tool.from_function(
        func=search.run,
        name="Search",
        description="useful for when you need to answer questions about current events"
    ),
    create_retriever_tool(
        db.as_retriever(), 
        "italy_travel",
        "Searches and returns documents regarding Italy."
    )
    ]

agent_executor = create_conversational_retrieval_agent(llm, tools, memory_key='chat_history', verbose=True)

In [19]:
memory

ConversationBufferMemory(chat_memory=InMemoryChatMessageHistory(messages=[]), return_messages=True, memory_key='chat_history')

In [20]:
agent_executor({"input": "what can I visit in India in 3 days?"})



> Entering new AgentExecutor chain...
In India, there are many amazing places you can visit in 3 days. Here are some popular destinations that you can consider:

1. Delhi: Explore the historical monuments like Red Fort, Qutub Minar, India Gate, and Humayun's Tomb. Don't miss visiting Chandni Chowk for shopping and trying delicious street food.

2. Agra: Visit the iconic Taj Mahal, Agra Fort, and Fatehpur Sikri. These historical monuments are must-visit attractions in Agra.

3. Jaipur: Known as the Pink City, Jaipur is famous for its majestic forts and palaces like Amber Fort, City Palace, and Hawa Mahal. You can also explore the local markets for traditional Rajasthani handicrafts.

4. Goa: If you prefer a more relaxed and beachy vibe, Goa is a great option. Enjoy the beautiful beaches, water sports, and vibrant nightlife in Goa.

5. Kerala: Experience the serene backwaters of Kerala by taking a houseboat cruise in Alleppey. You can also visit Munnar for its tea plantations and wildl

{'input': 'what can I visit in India in 3 days?',
 'chat_history': [HumanMessage(content='what can I visit in India in 3 days?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="In India, there are many amazing places you can visit in 3 days. Here are some popular destinations that you can consider:\n\n1. Delhi: Explore the historical monuments like Red Fort, Qutub Minar, India Gate, and Humayun's Tomb. Don't miss visiting Chandni Chowk for shopping and trying delicious street food.\n\n2. Agra: Visit the iconic Taj Mahal, Agra Fort, and Fatehpur Sikri. These historical monuments are must-visit attractions in Agra.\n\n3. Jaipur: Known as the Pink City, Jaipur is famous for its majestic forts and palaces like Amber Fort, City Palace, and Hawa Mahal. You can also explore the local markets for traditional Rajasthani handicrafts.\n\n4. Goa: If you prefer a more relaxed and beachy vibe, Goa is a great option. Enjoy the beautiful beaches, water sports, and vibrant nightlife i

In [21]:
agent_executor({"input": "what is the current wheather in Delhi ?"})



> Entering new AgentExecutor chain...

Invoking: `Search` with `current weather in Delhi`


{'type': 'weather_result', 'temperature': '81', 'unit': 'Fahrenheit', 'precipitation': '0%', 'humidity': '28%', 'wind': '7 mph', 'location': 'Delhi, India', 'date': 'Sunday 6:00 PM', 'weather': 'Clear'}The current weather in Delhi, India is as follows:
- Temperature: 81°F
- Precipitation: 0%
- Humidity: 28%
- Wind: 7 mph
- Weather: Clear

This information was last updated on Sunday at 6:00 PM.

> Finished chain.


{'input': 'what is the current wheather in Delhi ?',
 'chat_history': [HumanMessage(content='what can I visit in India in 3 days?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="In India, there are many amazing places you can visit in 3 days. Here are some popular destinations that you can consider:\n\n1. Delhi: Explore the historical monuments like Red Fort, Qutub Minar, India Gate, and Humayun's Tomb. Don't miss visiting Chandni Chowk for shopping and trying delicious street food.\n\n2. Agra: Visit the iconic Taj Mahal, Agra Fort, and Fatehpur Sikri. These historical monuments are must-visit attractions in Agra.\n\n3. Jaipur: Known as the Pink City, Jaipur is famous for its majestic forts and palaces like Amber Fort, City Palace, and Hawa Mahal. You can also explore the local markets for traditional Rajasthani handicrafts.\n\n4. Goa: If you prefer a more relaxed and beachy vibe, Goa is a great option. Enjoy the beautiful beaches, water sports, and vibrant nightlif

In [22]:
agent_executor({"input": "I'm travelling to Italy, can you give me some suggestions of the main attractions?"})



> Entering new AgentExecutor chain...

Invoking: `italy_travel` with `{'query': 'main attractions in Italy'}`


ITALY
MINUBE TRAVEL GUIDE
The best must-see places for your travels, all
discovered by real minube users. Enjoy!

What to see
 in Italy
Page 10
58
Gardens
Parco Sempione
 
Patricia M. Gómez:
 
This is one of the most important
parks in Milan. 
It was wonderful to visit it this winter all
covered in snow and then this spring for our first "picnic". The
park is very well kept and clean. 
You can easily go to
workout, walk or just have a little taste of green space in this
city, which can sometimes be gray. 
☎
 390 277 404 343
 - 
Piazza Castello, 20121 Milan, Italy, 
Milan
59
Exhibitions
Gulf of Naples
 
Carlos Millán Gómez:
 
Is there anywhere in the world
more beautiful than the Gulf of Naples? I would say no. It is
the very center of Paradise (perhaps it's also by the door of
hell, yes, because Mount Vesuvius could erupt at any time).
The Gulf smells like sea, lemon, fish,

{'input': "I'm travelling to Italy, can you give me some suggestions of the main attractions?",
 'chat_history': [HumanMessage(content='what can I visit in India in 3 days?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="In India, there are many amazing places you can visit in 3 days. Here are some popular destinations that you can consider:\n\n1. Delhi: Explore the historical monuments like Red Fort, Qutub Minar, India Gate, and Humayun's Tomb. Don't miss visiting Chandni Chowk for shopping and trying delicious street food.\n\n2. Agra: Visit the iconic Taj Mahal, Agra Fort, and Fatehpur Sikri. These historical monuments are must-visit attractions in Agra.\n\n3. Jaipur: Known as the Pink City, Jaipur is famous for its majestic forts and palaces like Amber Fort, City Palace, and Hawa Mahal. You can also explore the local markets for traditional Rajasthani handicrafts.\n\n4. Goa: If you prefer a more relaxed and beachy vibe, Goa is a great option. Enjoy the beautiful 